# 2-1회차 실습 | 펭귄 데이터로 진짜 실력 확인하기

1-2때 342마리로 94.44%, 트리는 100%까지 나왔음.
근데 그거 "이미 본 데이터" 점수임. 새 펭귄한테도 그럴지는 모름.

오늘 그거 확인함: Train/Test Split -> 교차검증 -> GridSearchCV -> 최종 평가.


---
## 실습 규칙

1. `[예측]` 있으면 실행 전에 먼저 적을 것
2. 숫자 나오면 "왜 이렇게 나왔나" 한 줄 적을 것


---
## 변수 이름 규약 (문제 노트북 전용)

뒷부분은 앞에서 만든 변수를 그대로 가져다 씀.
**이름을 아래와 똑같이** 지을 것. 다르게 지으면 뒤에서 에러남.

| 만드는 곳 | 변수 이름 | 담기는 것 |
|-----------|----------|----------|
| 준비 | `meas`, `peng` | 측정값 컬럼 리스트 / 342마리 표 (1-2와 동일) |
| 준비 | `X`, `y` | 특성 배열 / 정답 배열 |
| Q3 | `X_train, X_test, y_train, y_test` | 8:2 분리 결과 |
| Q4 | `dt`, `acc_split` | 모델 / 시험 정확도 |
| Q8 | `fold_scores` | 5-Fold 각각의 정확도 리스트 |
| Q11 | `grid` | GridSearchCV 객체 |
| Q12 | `best_model`, `final_acc` | 최적 모델 / 최종 홀드아웃 정확도 |

> 헷갈리면 해설 노트북 변수 이름 그대로 따라 쓸 것.


In [1]:
# 1-2회차에서 썼던 것과 똑같은 준비 과정입니다.
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings

# LogisticRegression이 반복 횟수 경고를 자주 띄우는데,
# 오늘은 스케일링을 배우기 전이라 어쩔 수 없이 나오는 경고입니다. 결과에는 영향 없습니다.
warnings.filterwarnings("ignore", category=UserWarning)

candidates = ["AppleGothic", "Malgun Gothic", "NanumGothic", "NanumBarunGothic"]
installed = {f.name for f in fm.fontManager.ttflist}
chosen = next((c for c in candidates if c in installed), None)
if chosen:
    plt.rcParams["font.family"] = chosen
plt.rcParams["axes.unicode_minus"] = False

# 데이터 로드 + 1-2회차와 동일하게 측정값 결측만 제거 (342마리)
peng_raw = sns.load_dataset("penguins")
meas = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
peng = peng_raw.dropna(subset=meas).reset_index(drop=True)

X = peng[meas].values
y = peng["species"].values

print(f"펭귄 {len(peng)}마리, 특성 {X.shape[1]}개")
print(f"품종: {sorted(set(y))}")


펭귄 342마리, 특성 4개
품종: ['Adelie', 'Chinstrap', 'Gentoo']


---
## Part 1. 훈련=평가 착시 다시 보기

### Q1. 전체 데이터로 학습하고 같은 데이터로 채점하면?

**[예측]** 몇 % 나올 것 같음? ______%


In [7]:
# TODO: 전체 데이터(X, y)로 트리를 학습시키고,
#       같은 X로 예측해서 y와 비교한 정확도를 구하세요.
# 힌트: DecisionTreeClassifier().fit(X, y) 뒤에 .predict(X)

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# 데이터 로드 + 1-2회차와 동일하게 측정값 결측만 제거 (342마리)
import seaborn as sns
peng_raw = sns.load_dataset("penguins")
meas = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
peng = peng_raw.dropna(subset=meas).reset_index(drop=True)

X = peng[meas].values
y = peng["species"].values

# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, stratify=y, random_state=42
# )

dt = DecisionTreeClassifier(random_state=18) #모델객채생성
dt.fit(X, y) ##학습
pred = dt.predict(X) ## 예측
acc = accuracy_score(y, pred)
acc


1.0

### Q2. 그럼 어떻게 진짜 실력을 재나요?

방법은 하나입니다. **모델이 한 번도 못 본 데이터**로 시험을 봐야 합니다.

그러려면 처음부터 데이터를 두 덩어리로 나눠야 합니다.

| 덩어리 | 역할 | 비유 |
|--------|------|------|
| Train (훈련) | 모델이 패턴을 배우는 데이터 | 교과서 |
| Test (시험) | 학습 때 한 번도 안 보여준, 마지막 채점용 데이터 | 봉인된 시험지 |

**중요한 규칙: Test 데이터는 학습 중에 절대 들여다보면 안 됩니다.**
성능이 궁금하다고 미리 열어 보는 순간, 그 순간부터 이 시험지는 무효입니다.
(이런 행위를 **데이터 누수, Data Leakage**라고 부릅니다 — 1-2회차 Q13의 island 문제와 본질이 비슷합니다.)


---
## Part 2. Train / Test Split

### Q3. 데이터를 8:2로 나누기

`test_size=0.2`, `stratify=y`, `random_state=7` 씀.

**[예측]** Train/Test 각각 몇 마리? ______ / ______


In [49]:
# TODO: train_test_split으로 X, y를 8:2로 나누세요.
# 힌트: test_size=0.2, stratify=y, random_state=7

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20%를 시험용으로
    stratify=y,         # 종 비율을 원본과 동일하게 유지
    random_state= 7     # 결과 고정 (재현성)
)

print(f"Train: {X_train.shape[0]}마리")
print(f"Train: {X_test.shape[0]}마리")

# stratify가 실제로 비율을 지켰는지 확인
import pandas as pd
print("\n원본 비율: ", pd.Series(y).value_counts(normalize=True).round(3).to_dict())
print("Train 비율: ", pd.Series(y_train).value_counts(normalize=True).round(3).to_dict())
print("Test 비율: ", pd.Series(y_test).value_counts(normalize=True).round(3).to_dict())

Train: 273마리
Train: 69마리

원본 비율:  {'Adelie': 0.442, 'Gentoo': 0.36, 'Chinstrap': 0.199}
Train 비율:  {'Adelie': 0.443, 'Gentoo': 0.359, 'Chinstrap': 0.198}
Test 비율:  {'Adelie': 0.435, 'Gentoo': 0.362, 'Chinstrap': 0.203}


### Q4. 진짜 시험 점수 확인

**[예측]** Q1의 100%보다 낮을까? 몇 %일 것 같음? ______%


In [ ]:
# TODO: X_train, y_train으로 트리를 학습시키고,
#       X_test로 예측해서 y_test와 비교한 정확도(acc_split)를 구하세요.



### Q5. 근데 91.30%, 믿어도 됨?

**[예측]** random_state 다른 숫자로 바꾸면 정확도 똑같이 나올까, 달라질까?


In [28]:
# TODO: random_state를 [51, 92, 14, 71, 60] 로 바꿔가며
#       각각 train_test_split -> 학습 -> 정확도를 출력하세요.
# 힌트: for 문 안에서 매번 새로 split 하고 새로 학습하세요.

random_state_test = [51, 92, 14, 71, 60]
for i in random_state_test:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=i
    )

    dt = DecisionTreeClassifier(random_state=7)
    dt.fit(X_train, y_train)
    print(f"DecisionTreeClassifier random_state={i}정확도: {accuracy_score(y_test, dt.predict(X_test)):.3f}")
    



DecisionTreeClassifier random_state=51정확도: 0.928
DecisionTreeClassifier random_state=92정확도: 0.942
DecisionTreeClassifier random_state=14정확도: 0.957
DecisionTreeClassifier random_state=71정확도: 0.971
DecisionTreeClassifier random_state=60정확도: 0.942


---
## Part 3. 모델 갈아 끼우기

### 오늘 처음 나오는 모델 둘 (한 줄 설명만)

| 모델 | 한 줄 설명 |
|------|-----------|
| `DecisionTreeClassifier` | 질문 계속 던져서 조건별로 갈라나가는 모델 |
| `LogisticRegression` | 클래스별 확률 계산해서 제일 높은 걸 고르는 모델 |
| `KNeighborsClassifier` (KNN) | 가까운 이웃 k개 다수결로 정하는 모델 |

원리는 몰라도 됨. 오늘은 "모델 바꿔도 fit→predict→score는 똑같다"만 체험.

### Q6. 모델 셋 비교하기 (DecisionTree / LogReg / KNN)

**[예측]** 셋 중 뭐가 제일 잘할 것 같음? 뭐가 제일 못할 것 같음? (감으로 찍어볼 것)


In [75]:
# TODO: DecisionTree, LogisticRegression(max_iter=3000), KNeighborsClassifier(5)
#        셋을 X_train, y_train으로 학습시키고 X_test 정확도를 비교하세요.
# 힌트: 딕셔너리에 넣고 for 문으로 돌리면 편합니다.



from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score

# 데이터 로드 + 1-2회차와 동일하게 측정값 결측만 제거 (342마리)
import seaborn as sns
peng_raw = sns.load_dataset("penguins")
meas = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
peng = peng_raw.dropna(subset=meas).reset_index(drop=True)

X = peng[meas].values
y = peng["species"].values

# 학습 및 테스트 데이터 분리 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 여러 모델을 교차검증으로 공정하게 비교 준비
models = {
    "DecisionTree": DecisionTreeClassifier(random_state=18, max_depth=5),
    "LogisticRegression": LogisticRegression(max_iter=3000, random_state=18),
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5)
}

print(f"{'모델':<20s}  {'test 정확도':>10s}  {'평균':>20s} {'표준편차':>10s}")
print("-" * 80)
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    print(f"{name:<20s}: {scores.round(3)} {scores.mean():>3f}  {scores.std():3f}")




모델                      test 정확도                    평균       표준편차
--------------------------------------------------------------------------------
DecisionTree        : [0.945 0.964 0.982 0.963 1.   ] 0.970774  0.018597
LogisticRegression  : [0.982 0.964 0.982 0.981 1.   ] 0.981751  0.011500
KNN (k=5)           : [0.782 0.764 0.709 0.722 0.685] 0.732391  0.035480


---
## Part 4. 교차검증 (Cross-Validation)

### Q7. K-Fold 직접 구현하기

용어 정리: Train(교과서) / **Validation**(모의고사, K-Fold 안에서 반복 사용) / Test(수능, 딱 1회).

Train 안에서 5등분, 매번 1등분은 Validation/나머지는 Train으로 5번 반복.
**Test는 여전히 안 건드림.**

**[예측]** 5개 점수가 거의 같을 것 같음, 꽤 다를 것 같음?


In [ ]:
# TODO: StratifiedKFold(n_splits=5)로 X_train, y_train을 5등분해서
#        각 fold의 정확도를 fold_scores 리스트에 모으고, 평균/표준편차를 출력하세요.
# 힌트: kf.split(X_train, y_train) 은 (train_idx, val_idx) 쌍을 5번 돌려줍니다.


from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
import pandas as pd

# 데이터 로드 [측정값 결측만 제거 (342마리)]
import seaborn as sns
peng_raw = sns.load_dataset("penguins")
meas = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
peng = peng_raw.dropna(subset=meas).reset_index(drop=True)

X = peng[meas].values
y = peng["species"].values

# 학습 및 테스트 데이터 분리 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=18)

fold_scores = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X,y), 1):
    model = DecisionTreeClassifier(random_state=18)
    model.fit(X[train_idx], y[train_idx])
    score = accuracy_score(y[test_idx], model.predict(X[test_idx]))
    fold_scores.append((fold, score))

df_scores = pd.DataFrame(fold_scores, columns=["Fold","Accuracy"])
display(df_scores)
print(f"평균={df_scores['Accuracy'].mean():.3f}, 표준편차={df_scores['Accuracy'].std():.3f}")



,Fold,Accuracy
0,1,0.971014
1,2,0.956522
2,3,1.000000
3,4,0.970588
4,5,0.970588


평균=0.974, 표준편차=0.016


### Q8. for문 대신 한 줄로: cross_val_score

**[예측]** Q7 평균과 똑같이 나올까?


In [ ]:
# TODO: cross_val_score를 사용해서 Q7과 같은 결과를 한 줄로 얻으세요.
# 힌트: cross_val_score(모델, X_train, y_train, cv=5)



from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
#from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score
import pandas as pd

# 데이터 로드 [측정값 결측만 제거 (342마리)]
import seaborn as sns
peng_raw = sns.load_dataset("penguins")
meas = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
peng = peng_raw.dropna(subset=meas).reset_index(drop=True)

X = peng[meas].values
y = peng["species"].values

# 학습 및 테스트 데이터 분리 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

########################################################
# DecisionTree에 5-Fold 교차검증 적용
dt = DecisionTreeClassifier(random_state=18)
scores = cross_val_score(dt,
                        X_train,
                        y_train,
                        cv=4,
                        scoring="accuracy")
########################################################

print(f"각 Fold 정확도: {scores.round(3)}")
print(f"평균 정확도:    {scores.mean():.3f} ± {scores.std():.3f}")
print()




각 Fold 정확도: [0.957 0.941 0.971 1.   ]
평균 정확도:    0.967 ± 0.022



### Q9. 세 모델, 교차검증으로 다시 비교

**[예측]** Q6이랑 순위 같을까, 바뀔까?


In [ ]:
# TODO: Q6에서 만든 models 딕셔너리를 그대로 써서,
#        각 모델의 교차검증 평균/표준편차를 출력하세요.



---
## Part 5. GridSearchCV

### Q10. max_depth 바꾸면 성능 바뀔까?

**[예측]** 1, 2, 3, 4, None 순서로 늘리면 정확도 계속 오를까, 어디서 꺾일까?


In [ ]:
# TODO: max_depth를 [1, 2, 3, 4, None] 으로 바꿔가며
#        cross_val_score 평균/표준편차를 표로 정리하고, 그래프로 그리세요.



### Q11. GridSearchCV로 자동화하기

**중요:** GridSearchCV는 '설계자'가 아니라 '탐색기'임.
내가 넣은 후보 안에서만 비교함. 후보 범위 잘못 잡으면 아무리 돌려도 소용없음.

**[예측]** 최적 max_depth 몇일 것 같음? (Q10 보고 답해볼 것)


In [ ]:
# TODO: param_grid = {"max_depth": [1, 2, 3, 4, None]} 로
#        GridSearchCV를 만들고 X_train, y_train에 fit 하세요.
# 힌트: GridSearchCV(모델, param_grid, cv=5, scoring="accuracy")



---
## Part 6. 최종 홀드아웃 평가

### Q12. 진짜 마지막 시험

**[예측]** CV 점수(0.9708)랑 최종 test 점수 비슷할까, 많이 다를까?


In [ ]:
# TODO: grid.best_estimator_ 로 X_test를 예측하고, y_test와 비교한
#        final_acc를 구하세요. Test 데이터는 지금 처음 씁니다.



---
## 오늘 정리

빈칸 채워볼 것.

| 방법 | 정확도 |
|------|--------|
| 전체로 학습+평가 (cheat) | 1.0000 |
| Train/Test 분리 | ______ |
| 5-Fold 평균 (DecisionTree) | ______ |
| 5-Fold 평균 (LogisticRegression) | ______ |
| GridSearchCV 최적 CV 점수 | ______ |
| 최종 홀드아웃 test 점수 | ______ |

다음 시간: 전처리(스케일링/인코딩), Pipeline, Confusion Matrix/Precision/Recall/F1
